# `diffusion_lib` — Use Case Walkthrough

**Authors:** Iván Domínguez Hernández · Lucas Miranda López  
**Tutor:** Alberto Suárez González  
**Course:** Aprendizaje Automático III — Grado en Ciencia e Ingeniería de Datos, UAM

---

This notebook walks through **all five use cases** of the `diffusion_lib` package.
Each section maps directly to a use case from the Software Design chapter:

| # | Use Case | Key components |
|---|---|---|
| CU-01 | **Score model training** | `DiffusionModel`, `VEProcess`/`VPProcess`, `UNet` |
| CU-02 | **Unconditional generation** | `EulerMaruyama`, `PredictorCorrector`, `ProbabilityFlowODE` |
| CU-03 | **Class-conditioned synthesis (CFG)** | `ConditionalUNet`, `CFGWrapper` |
| CU-04 | **Inpainting** | `ImputationSampler` |
| CU-05 | **Quality evaluation** | `BPDEvaluator`, `FIDISEvaluator` |


---
## 0 · Environment Setup

Install `diffusion_lib` and its dependencies before running any section.


In [1]:
# !pip install -e .              # from repo root
# !pip install torch torchvision numpy matplotlib tqdm


In [2]:
import torch
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from diffusion_lib.model import DiffusionModel
from diffusion_lib.processes.ve import VEProcess
from diffusion_lib.processes.vp import VPProcess
from diffusion_lib.schedules.linear import LinearSchedule
from diffusion_lib.schedules.cosine import CosineSchedule
from diffusion_lib.score_models.unconditional import UnconditionalUNet
from diffusion_lib.score_models.conditional  import ConditionalUNet
from diffusion_lib.score_models.cfg_wrapper  import CFGWrapper
from diffusion_lib.samplers.euler_maruyama       import EulerMaruyamaSampler
from diffusion_lib.samplers.predictor_corrector  import PredictorCorrectorSampler
from diffusion_lib.samplers.probability_flow_ode import ProbabilityFlowODESampler
from diffusion_lib.samplers.imputation           import ImputationSampler
from diffusion_lib.metrics.bpd    import BPDEvaluator
from diffusion_lib.metrics.fid_is import FIDISEvaluator

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running on: {DEVICE}')


ImportError: cannot import name 'DiffusionModel' from 'diffusion_lib.model' (/Users/ivandominguez/Desktop/practicas3segundocuatri/apaut3/proyecto/proyecto_AAIII_02_diffusion_models/diffusion_lib/model.py)

In [ ]:
# Global hyper-parameters
IMG_SIZE   = 32
CHANNELS   = 3
T_STEPS    = 1000
BATCH_SIZE = 128
CKPT_DIR   = Path('checkpoints')
CKPT_DIR.mkdir(exist_ok=True)

print(f'Image size : {IMG_SIZE}x{IMG_SIZE}x{CHANNELS}')
print(f'Time steps : {T_STEPS}')


---
## CU-01 · Score Model Training

**Goal:** configure a diffusion process (VE or VP), instantiate a U-Net score model, and run the denoising score-matching training loop.

At each training iteration $t \sim \mathcal{U}\{1,\ldots,T\}$:
1. **Forward process** injects Gaussian noise into $\mathbf{x}_0$.
2. The U-Net predicts the injected noise $\boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)$.
3. **Score-matching loss** is minimised:
$$\mathcal{L}(\theta)=\mathbb{E}_{t,\mathbf{x}_0,\boldsymbol{\epsilon}}\bigl[\|\boldsymbol{\epsilon}_\theta(\mathbf{x}_t,t)-\boldsymbol{\epsilon}\|_2^2\bigr]$$


In [ ]:
# 1.1  Dataset (CIFAR-10)
transform = T.Compose([
    T.ToTensor(),
    T.Normalize([0.5]*CHANNELS, [0.5]*CHANNELS),
])
train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True
)
print(f'Training samples: {len(train_dataset):,}  |  Batches/epoch: {len(train_loader)}')


In [ ]:
# 1.2a  Variance Exploding (VE) process
#   Forward:   x_{t+1} = x_t + sigma^t * z
#   Backward:  x_{t-1} = x_t + sigma^{2t} * score * dt + sigma^t * sqrt(dt) * z
ve_process = VEProcess(
    sigma_min=0.01,
    sigma_max=50.0,
    T=T_STEPS,
    device=DEVICE,
)
print(f'VE process  |  sigma: [{ve_process.sigma_min}, {ve_process.sigma_max}]')


In [ ]:
# 1.2b  Variance Preserving (VP) process with cosine schedule
#   Forward (Ornstein-Uhlenbeck):
#     x_{t+1} = x_t - (1/2)*beta(t)*x_t*dt + sqrt(beta(t))*z
#
#   Cosine schedule avoids premature noise saturation at low resolution
#   (Nichol & Dhariwal 2021).
cosine_schedule = CosineSchedule(T=T_STEPS, s=0.008, device=DEVICE)
vp_process      = VPProcess(schedule=cosine_schedule, T=T_STEPS, device=DEVICE)

# Visualise schedule
t_range = torch.linspace(0, 1, 200)
alphas  = cosine_schedule.alpha(t_range).cpu().numpy()
betas   = cosine_schedule.beta(t_range).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(t_range, alphas, color='steelblue', lw=2)
axes[0].set_title(r'Cumulative $\alpha_t$ (cosine)')
axes[0].set_xlabel('t'); axes[0].set_ylabel(r'$\alpha_t$')
axes[1].plot(t_range, betas, color='firebrick', lw=2)
axes[1].set_title(r'Noise $\beta(t)$ (cosine)')
axes[1].set_xlabel('t'); axes[1].set_ylabel(r'$\beta(t)$')
plt.tight_layout(); plt.show()
print(f'beta_min={betas.min():.4f}  beta_max={betas.max():.4f}')


In [ ]:
# 1.3  Visualise the forward process (noise injection)
sample_img = train_dataset[0][0].unsqueeze(0).to(DEVICE)
timesteps  = [0, 100, 250, 500, 750, 999]

fig, axes = plt.subplots(1, len(timesteps), figsize=(13, 2.5))
for ax, t_idx in zip(axes, timesteps):
    t_tensor = torch.tensor([t_idx], device=DEVICE)
    noisy    = vp_process.forward(sample_img, t_tensor)  # RF-05
    ax.imshow(noisy[0].permute(1,2,0).cpu().clamp(-1,1).numpy()*0.5+0.5)
    ax.set_title(f't={t_idx}'); ax.axis('off')

fig.suptitle('VP forward process: progressive noise injection', fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# 1.4  U-Net score model (RF-02)
#   - Adaptive input channels (grayscale or RGB)
#   - Sinusoidal time-step embeddings injected in every residual block
#   - Multi-head self-attention at 16x16 resolution
score_model = UnconditionalUNet(
    in_channels=CHANNELS,
    base_channels=128,
    channel_multipliers=[1, 2, 2, 2],
    num_res_blocks=2,
    attention_resolutions=[16],
    dropout=0.1,
    time_embed_dim=512,
).to(DEVICE)

print(f'UnconditionalUNet  |  Parameters: {sum(p.numel() for p in score_model.parameters()):,}')


In [ ]:
# 1.5  DiffusionModel: integrates process + score model  (RF-01)
model = DiffusionModel(process=vp_process, score_model=score_model, device=DEVICE)
print(model)


In [ ]:
# 1.6  Training loop (RF-06)
#
#  train_epoch() per iteration:
#    1. Samples t ~ U{1,...,T} for each image in the batch
#    2. Computes x_t via the forward process
#    3. Predicts injected noise with the U-Net
#    4. Minimises denoising score-matching loss
#    5. Backprop + optimiser step

NUM_EPOCHS    = 3       # set to 500+ for production quality
LEARNING_RATE = 2e-4

optimiser    = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimiser, T_max=NUM_EPOCHS * len(train_loader)
)
train_losses = []

for epoch in range(1, NUM_EPOCHS + 1):
    loss = model.train_epoch(
        dataloader=train_loader,
        optimiser=optimiser,
        lr_scheduler=lr_scheduler,
    )  # RF-06
    train_losses.append(loss)
    print(f'Epoch {epoch:>3}/{NUM_EPOCHS}  |  loss: {loss:.5f}')

ckpt_path = CKPT_DIR / 'vp_cosine_uncond.pt'
model.save_checkpoint(ckpt_path)
print(f'Checkpoint saved -> {ckpt_path}')


In [ ]:
# 1.7  Training loss curve
plt.figure(figsize=(7, 3.5))
plt.plot(train_losses, marker='o', lw=2, color='steelblue')
plt.xlabel('Epoch'); plt.ylabel('Score-matching loss')
plt.title('CU-01 - Training curve (VP + cosine schedule)')
plt.grid(alpha=0.4); plt.tight_layout(); plt.show()


---
## CU-02 · Unconditional Image Generation

**Goal:** synthesise new images by integrating the **backward SDE / ODE** from pure Gaussian noise.

Three samplers are compared (RF-07):

| Sampler | Stochastic | Evals/step | BPD | Quality at low N |
|---|---|---|---|---|
| Euler-Maruyama | Yes | 1 | No | Low |
| Predictor-Corrector | Yes | M+1 | No | Medium-High |
| Probability Flow ODE | No | 1 | Yes | Medium* |

Backward VP SDE (Euler-Maruyama discretisation):
$$\mathbf{x}_{t-1}=\mathbf{x}_t+\left[\tfrac{1}{2}\beta(t)\mathbf{x}_t+\beta(t)\nabla_{\mathbf{x}}\log p_t\right]\Delta t+\sqrt{\beta(t)\Delta t}\,\mathbf{z}_t$$


In [ ]:
# 2.1  Load checkpoint
model.load_checkpoint(CKPT_DIR / 'vp_cosine_uncond.pt')
model.eval()
print('Checkpoint loaded.')


In [ ]:
# 2.2  Euler-Maruyama sampler
em_sampler = EulerMaruyamaSampler(model=model, num_steps=500, device=DEVICE)

with torch.no_grad():
    samples_em = em_sampler.sample(shape=(16, CHANNELS, IMG_SIZE, IMG_SIZE))

print(f'EM samples: {samples_em.shape}')


In [ ]:
# 2.3  Predictor-Corrector sampler
#   Each predictor step (EM) is followed by M Langevin corrector steps
#   that re-project the sample onto the marginal p_{t+1}.
pc_sampler = PredictorCorrectorSampler(
    model=model, num_steps=500,
    corrector_steps=1,
    snr=0.16,        # target SNR for automatic corrector step size (Song 2021)
    device=DEVICE,
)

with torch.no_grad():
    samples_pc = pc_sampler.sample(shape=(16, CHANNELS, IMG_SIZE, IMG_SIZE))

print(f'PC samples: {samples_pc.shape}')


In [ ]:
# 2.4  Probability Flow ODE sampler
#   Deterministic ODE sharing the same marginals as the backward SDE.
#   Enables exact likelihood computation (BPD) via change-of-variables.
#   dx/dt = f(x,t) - (1/2)*g(t)^2 * score
ode_sampler = ProbabilityFlowODESampler(
    model=model, num_steps=500,
    method='euler',  # or 'heun' / 'rk45' for higher accuracy
    device=DEVICE,
)

with torch.no_grad():
    samples_ode = ode_sampler.sample(shape=(16, CHANNELS, IMG_SIZE, IMG_SIZE))

print(f'ODE samples: {samples_ode.shape}')


In [ ]:
# 2.5  Visual comparison
def show_grid(tensor, title, nrow=8):
    imgs = (tensor.clamp(-1, 1) * 0.5 + 0.5).cpu()
    grid = torchvision.utils.make_grid(imgs, nrow=nrow, padding=2)
    plt.figure(figsize=(10, 2.5))
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.title(title, fontsize=12); plt.axis('off')
    plt.tight_layout(); plt.show()

show_grid(samples_em,  'CU-02 - Euler-Maruyama (unconditional)')
show_grid(samples_pc,  'CU-02 - Predictor-Corrector (unconditional)')
show_grid(samples_ode, 'CU-02 - Probability Flow ODE (unconditional)')


---
## CU-03 · Class-Conditioned Synthesis (Classifier-Free Guidance)

**Goal:** generate images of a specific class using Classifier-Free Guidance (CFG).

### Mechanism (Ho & Salimans 2022)

One model is trained for both conditional and unconditional prediction.  
At inference the effective score is extrapolated:
$$\tilde{\boldsymbol{\epsilon}}_\theta(\mathbf{x}_t,t\mid y)=\boldsymbol{\epsilon}_\theta(\mathbf{x}_t,t\mid\emptyset)+w\bigl[\boldsymbol{\epsilon}_\theta(\mathbf{x}_t,t\mid y)-\boldsymbol{\epsilon}_\theta(\mathbf{x}_t,t\mid\emptyset)\bigr]$$

$w\in[1,10]$: higher $w$ improves class fidelity at the cost of diversity.


In [ ]:
# 3.1  Conditional U-Net: adds learned class embedding to time embedding (RF-02, RF-08)
NUM_CLASSES = 10

cond_score_model = ConditionalUNet(
    in_channels=CHANNELS,
    num_classes=NUM_CLASSES,
    base_channels=128,
    channel_multipliers=[1, 2, 2, 2],
    num_res_blocks=2,
    attention_resolutions=[16],
    dropout=0.1,
    time_embed_dim=512,
    uncond_prob=0.1,  # probability of dropping label -> trains unconditional branch
).to(DEVICE)

print(f'ConditionalUNet  |  Parameters: {sum(p.numel() for p in cond_score_model.parameters()):,}')


In [ ]:
# 3.2  CFGWrapper: combines two forward passes at inference for guided sampling
cfg_model = CFGWrapper(score_model=cond_score_model, guidance_scale=7.5)

cond_model = DiffusionModel(process=vp_process, score_model=cfg_model, device=DEVICE)
print('CFGWrapper ready.  guidance_scale =', cfg_model.guidance_scale)


In [ ]:
# 3.3  Train conditional model (RF-06, RF-08)
#   Class labels are randomly dropped with probability uncond_prob
#   to jointly train conditional and unconditional branches.
NUM_EPOCHS_COND = 3
opt_cond   = torch.optim.Adam(cond_model.parameters(), lr=2e-4)
cond_losses = []

for epoch in range(1, NUM_EPOCHS_COND + 1):
    loss = cond_model.train_epoch(
        dataloader=train_loader, optimiser=opt_cond, use_labels=True
    )
    cond_losses.append(loss)
    print(f'Epoch {epoch}/{NUM_EPOCHS_COND}  |  cond loss: {loss:.5f}')

ckpt_cond = CKPT_DIR / 'vp_cosine_cond_cfg.pt'
cond_model.save_checkpoint(ckpt_cond)
print(f'Checkpoint saved -> {ckpt_cond}')


In [ ]:
# 3.4  Guided generation for all 10 CIFAR-10 classes
CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer',
                   'dog','frog','horse','ship','truck']

cond_model.load_checkpoint(ckpt_cond)
cond_model.eval()

cfg_sampler = PredictorCorrectorSampler(
    model=cond_model, num_steps=500, corrector_steps=1, snr=0.16, device=DEVICE
)

N_PER_CLASS = 4
all_cond = []

with torch.no_grad():
    for cls in range(NUM_CLASSES):
        labels  = torch.full((N_PER_CLASS,), cls, dtype=torch.long, device=DEVICE)
        samples = cfg_sampler.sample(
            shape=(N_PER_CLASS, CHANNELS, IMG_SIZE, IMG_SIZE),
            labels=labels,    # RF-08
        )
        all_cond.append(samples)

all_cond = torch.cat(all_cond, dim=0)
print(f'Generated {all_cond.shape[0]} conditioned images ({N_PER_CLASS} per class).')


In [ ]:
# 3.5  Visualise per-class samples
imgs = (all_cond.clamp(-1, 1) * 0.5 + 0.5).cpu()
grid = torchvision.utils.make_grid(imgs, nrow=N_PER_CLASS, padding=3)

fig, ax = plt.subplots(figsize=(5, 14))
ax.imshow(grid.permute(1, 2, 0).numpy())
ax.set_title('CU-03 - Classifier-Free Guidance  (w=7.5)', fontsize=11)
ax.axis('off')
cell_h = grid.shape[1] / NUM_CLASSES
for i, name in enumerate(CIFAR10_CLASSES):
    ax.text(-3, (i+0.5)*cell_h, name, va='center', ha='right', fontsize=9)
plt.tight_layout(); plt.show()


In [ ]:
# 3.6  Guidance scale sweep on class 'cat'
TARGET_CLASS   = 3
sweep_w        = [1.0, 3.0, 5.0, 7.5, 10.0]
sweep_samples  = []

with torch.no_grad():
    for w in sweep_w:
        cfg_model.guidance_scale = w
        labels = torch.full((4,), TARGET_CLASS, dtype=torch.long, device=DEVICE)
        sweep_samples.append(
            cfg_sampler.sample(shape=(4, CHANNELS, IMG_SIZE, IMG_SIZE), labels=labels)
        )

fig, axes = plt.subplots(len(sweep_w), 4, figsize=(7, 10))
for row, (w, s) in enumerate(zip(sweep_w, sweep_samples)):
    imgs_w = (s.clamp(-1,1)*0.5+0.5).cpu()
    for col in range(4):
        axes[row][col].imshow(imgs_w[col].permute(1,2,0).numpy())
        axes[row][col].axis('off')
    axes[row][0].set_ylabel(f'w={w}', fontsize=9, rotation=0, labelpad=30)
fig.suptitle(f'CU-03 - Guidance sweep (class: {CIFAR10_CLASSES[TARGET_CLASS]})', fontsize=11)
plt.tight_layout(); plt.show()

cfg_model.guidance_scale = 7.5  # restore


---
## CU-04 · Inpainting (Masked Restoration)

**Goal:** reconstruct missing regions of an image while preserving all known pixels.

### Replacement strategy (RF-09)

Given binary mask $\mathbf{m}\in\{0,1\}^D$, at each backward step:

$$\mathbf{x}_{t-1}=\mathbf{m}\odot\mathbf{x}_{t-1,\text{fwd}}+(1-\mathbf{m})\odot\mathbf{x}_{t-1,\text{bwd}}$$

Known pixels are anchored via the forward process; unknown pixels are generated freely.


In [ ]:
# 4.1  Test images and three masking strategies
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform
)
N_INPAINT = 4
test_imgs  = torch.stack([test_dataset[i][0] for i in range(N_INPAINT)]).to(DEVICE)

H, W = IMG_SIZE, IMG_SIZE

mask_bottom = torch.ones(1, 1, H, W, device=DEVICE)
mask_bottom[:, :, H//2:, :] = 0                         # occlude bottom half

mask_center = torch.ones(1, 1, H, W, device=DEVICE)
mask_center[:, :, H//4:3*H//4, W//4:3*W//4] = 0        # central patch

torch.manual_seed(42)
mask_random = (torch.rand(1, 1, H, W, device=DEVICE) > 0.5).float()  # 50% dropout

masks = {'bottom_half': mask_bottom, 'center_patch': mask_center, 'random_50pct': mask_random}

fig, axes = plt.subplots(len(masks), N_INPAINT+1, figsize=(10, 7))
for row, (name, mask) in enumerate(masks.items()):
    corr = test_imgs * mask
    axes[row][0].imshow(mask[0,0].cpu().numpy(), cmap='gray')
    axes[row][0].set_title(f'Mask\n({name})', fontsize=8); axes[row][0].axis('off')
    for col in range(N_INPAINT):
        axes[row][col+1].imshow(
            (corr[col].permute(1,2,0).cpu().clamp(-1,1).numpy()*0.5+0.5)
        )
        axes[row][col+1].axis('off')
fig.suptitle('CU-04 - Corrupted inputs', fontsize=11)
plt.tight_layout(); plt.show()


In [ ]:
# 4.2  Inpainting with ImputationSampler (RF-09)
model.load_checkpoint(CKPT_DIR / 'vp_cosine_uncond.pt')
model.eval()

inpaint_results = {}

with torch.no_grad():
    for mask_name, mask in masks.items():
        imputer = ImputationSampler(model=model, num_steps=500, device=DEVICE)
        restored = imputer.sample(
            known_pixels=test_imgs,
            mask=mask.expand_as(test_imgs[:1]),
        )
        inpaint_results[mask_name] = restored
        print(f'Inpainting done: {mask_name}')


In [ ]:
# 4.3  Visual comparison: original | corrupted | restored
def to_np(t):
    return (t.permute(1,2,0).cpu().clamp(-1,1).numpy()*0.5+0.5)

for mask_name, restored in inpaint_results.items():
    corr = test_imgs * masks[mask_name]
    fig, axes = plt.subplots(3, N_INPAINT, figsize=(9, 7))
    for col in range(N_INPAINT):
        axes[0][col].imshow(to_np(test_imgs[col])); axes[0][col].axis('off')
        axes[1][col].imshow(to_np(corr[col]));      axes[1][col].axis('off')
        axes[2][col].imshow(to_np(restored[col]));  axes[2][col].axis('off')
    for row, lbl in enumerate(['Original', 'Corrupted', 'Restored']):
        axes[row][0].set_ylabel(lbl, fontsize=9, rotation=0, labelpad=45)
    fig.suptitle(f'CU-04 - Inpainting  ({mask_name})', fontsize=11)
    plt.tight_layout(); plt.show()


---
## CU-05 · Quality Evaluation (BPD, FID, IS)

**Goal:** quantitatively assess the model with three complementary metrics (RF-10, RF-11):

| Metric | Formula | Better |
|---|---|---|
| **BPD** | $-\frac{1}{D}\mathbb{E}[\log_2 p(\mathbf{x})]$ | Lower |
| **FID** | $\|\mu_r-\mu_g\|^2 + \mathrm{Tr}(\Sigma_r+\Sigma_g-2(\Sigma_r\Sigma_g)^{1/2})$ | Lower |
| **IS** | $\exp(\mathbb{E}[D_{\mathrm{KL}}(p(y|\mathbf{x})\|p(y))])$ | Higher |

DDPM reference on CIFAR-10: **FID~3.17 | IS~9.46 | BPD~3.75**


In [ ]:
# 5.1  Instantiate evaluators
bpd_evaluator    = BPDEvaluator(model=model, device=DEVICE)          # RF-10
fid_is_evaluator = FIDISEvaluator(device=DEVICE, num_features=2048)  # RF-11  InceptionV3 pool_3
print('Evaluators ready.')


In [ ]:
# 5.2  Bits Per Dimension
#   Encoded via Probability Flow ODE + instantaneous change-of-variables:
#     log p0(x0) = log pT(xT) + integral_0^T div(f_theta) dt
bpd_loader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(test_dataset, range(256)), batch_size=32
)
model.load_checkpoint(CKPT_DIR / 'vp_cosine_uncond.pt')
model.eval()

bpd_value = bpd_evaluator.compute(dataloader=bpd_loader, num_bits=8)
print(f'BPD = {bpd_value:.4f} bits/dim  (DDPM reference: ~3.75)')


In [ ]:
# 5.3  Generate batch for FID / IS  (increase N_EVAL to 10000 for publications)
N_EVAL = 1000;  N_REAL = 1000;  EVAL_BATCH = 50

em_eval = EulerMaruyamaSampler(model=model, num_steps=500, device=DEVICE)
gen_imgs = []

with torch.no_grad():
    for _ in range(N_EVAL // EVAL_BATCH):
        gen_imgs.append(
            em_eval.sample(shape=(EVAL_BATCH, CHANNELS, IMG_SIZE, IMG_SIZE)).cpu()
        )
gen_imgs  = torch.cat(gen_imgs, dim=0)
real_imgs = torch.stack([test_dataset[i][0] for i in range(N_REAL)])
print(f'Generated: {gen_imgs.shape[0]}  |  Real: {real_imgs.shape[0]}')


In [ ]:
# 5.4  Compute FID and IS
gen01  = (gen_imgs.clamp(-1,1)*0.5+0.5)
real01 = (real_imgs.clamp(-1,1)*0.5+0.5)

fid, is_m, is_s = fid_is_evaluator.compute(
    real_images=real01, generated_images=gen01, is_splits=10
)
print('-'*45)
print(f'  FID  = {fid:.2f}          (lower is better)')
print(f'  IS   = {is_m:.2f} +/- {is_s:.2f}   (higher is better)')
print(f'  BPD  = {bpd_value:.4f}      (lower is better)')
print('-'*45)
print('DDPM reference:  FID~3.17  IS~9.46  BPD~3.75')


In [ ]:
# 5.5  Sampler benchmark: compare all three samplers
samplers_bench = {
    'Euler-Maruyama'      : EulerMaruyamaSampler(model=model, num_steps=500, device=DEVICE),
    'Predictor-Corrector' : PredictorCorrectorSampler(model=model, num_steps=500, corrector_steps=1, snr=0.16, device=DEVICE),
    'Probability Flow ODE': ProbabilityFlowODESampler(model=model, num_steps=500, method='euler', device=DEVICE),
}

bench_results = {}
for name, sampler in samplers_bench.items():
    print(f'Evaluating {name} ...')
    batches = []
    with torch.no_grad():
        for _ in range(N_EVAL // EVAL_BATCH):
            batches.append(sampler.sample(shape=(EVAL_BATCH, CHANNELS, IMG_SIZE, IMG_SIZE)).cpu())
    g01 = (torch.cat(batches,0).clamp(-1,1)*0.5+0.5)
    fid_b, is_mb, is_sb = fid_is_evaluator.compute(real_images=real01, generated_images=g01, is_splits=10)
    bench_results[name] = {'FID': fid_b, 'IS_mean': is_mb, 'IS_std': is_sb}
    print(f'  FID={fid_b:.2f}  IS={is_mb:.2f}+/-{is_sb:.2f}')


In [ ]:
# 5.6  Bar chart: metric comparison
names  = list(bench_results.keys())
fids   = [bench_results[n]['FID']     for n in names]
is_ms  = [bench_results[n]['IS_mean'] for n in names]
is_ss  = [bench_results[n]['IS_std']  for n in names]
colors = ['steelblue', 'seagreen', 'firebrick']
x      = np.arange(len(names))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(x, fids, color=colors, alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(names, rotation=12, ha='right', fontsize=9)
axes[0].set_ylabel('FID  (lower = better)'); axes[0].set_title('Frechet Inception Distance')
axes[0].axhline(3.17, ls='--', color='black', lw=1, label='DDPM ref'); axes[0].legend(fontsize=8)

axes[1].bar(x, is_ms, yerr=is_ss, color=colors, alpha=0.85, capsize=5)
axes[1].set_xticks(x); axes[1].set_xticklabels(names, rotation=12, ha='right', fontsize=9)
axes[1].set_ylabel('IS  (higher = better)'); axes[1].set_title('Inception Score')
axes[1].axhline(9.46, ls='--', color='black', lw=1, label='DDPM ref'); axes[1].legend(fontsize=8)

fig.suptitle('CU-05 - Sampler benchmark (CIFAR-10)', fontsize=12)
plt.tight_layout(); plt.show()


---
## Summary

| Use Case | Key API | What was shown |
|---|---|---|
| **CU-01** Training | `DiffusionModel.train_epoch()` | VE/VP setup, forward visualisation, loss curve |
| **CU-02** Unconditional gen. | `EulerMaruyamaSampler`, `PredictorCorrectorSampler`, `ProbabilityFlowODESampler` | 3-sampler visual comparison |
| **CU-03** Conditional (CFG) | `ConditionalUNet`, `CFGWrapper` | Per-class synthesis; guidance scale sweep |
| **CU-04** Inpainting | `ImputationSampler` | 3 mask types; original/corrupted/restored |
| **CU-05** Quality metrics | `BPDEvaluator`, `FIDISEvaluator` | BPD, FID, IS; sampler benchmark vs DDPM |

**For production quality:** 500+ training epochs; 10 000+ images for FID; `heun` ODE solver for BPD.

---
*Aprendizaje Automatico III — UAM 2026*
